Setup

In [1]:
import sys
sys.path.insert(0, "..")

from src.rag.knowledge_base import DOCUMENTS, TOPICS
from src.rag.retriever import (
    get_or_create_collection,
    retrieve,
    get_context_for_query,
    reset_knowledge_base
)
from src.rag.qa_engine import answer_question, answer_batch

print(f"Knowledge base: {len(DOCUMENTS)} documents")
print(f"Topics covered: {list(TOPICS.keys())}")


Knowledge base: 10 documents
Topics covered: ['RSI', 'MACD', 'Bollinger Bands', 'Moving Averages', 'ATR', 'Volume', 'Probability', 'SHAP', 'News Sentiment', 'NIFTY50']


 Initialise the knowledge base

In [2]:
# First run downloads the embedding model (~80MB, one time only)
# Subsequent runs load from disk — instant
print("Initialising ChromaDB knowledge base...")
collection = get_or_create_collection()
print(f"Ready. {collection.count()} chunks indexed.")

Initialising ChromaDB knowledge base...


C:\Users\devgi\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Knowledge base loaded: 49 documents.
Ready. 49 chunks indexed.


Test retrieval

In [3]:
# Test that semantic search is working correctly
test_queries = [
    "what does RSI mean",
    "my stock has high volume today",
    "bands are getting narrow",
    "why is my probability low",
    "what is a death cross"
]

print("Testing semantic retrieval...")
print("─" * 55)

for query in test_queries:
    chunks = retrieve(query, n_results=2)
    print(f"\nQuery: '{query}'")
    for c in chunks:
        print(f"  → [{c['topic']}] {c['title']} "
              f"(distance={c['distance']:.3f})")

Testing semantic retrieval...
───────────────────────────────────────────────────────
Knowledge base loaded: 49 documents.

Query: 'what does RSI mean'
  → [RSI] What is RSI (Relative Strength Index)? (distance=0.228)
  → [RSI] What is RSI (Relative Strength Index)? (distance=0.504)
Knowledge base loaded: 49 documents.

Query: 'my stock has high volume today'
  → [Volume] What does trading volume mean? (distance=0.371)
  → [Volume] What does trading volume mean? (distance=0.378)
Knowledge base loaded: 49 documents.

Query: 'bands are getting narrow'
  → [Bollinger Bands] What are Bollinger Bands? (distance=0.499)
  → [Bollinger Bands] What are Bollinger Bands? (distance=0.573)
Knowledge base loaded: 49 documents.

Query: 'why is my probability low'
  → [Probability] What does the InvestIQ probability score mean? (distance=0.496)
  → [Probability] What does the InvestIQ probability score mean? (distance=0.570)
Knowledge base loaded: 49 documents.

Query: 'what is a death cross'
  → [Mov

RAG vs no-RAG comparison

In [5]:
# This is the key demonstration — show that RAG produces
# more grounded, consistent answers than LLM alone
import time
question = "What does it mean when RSI is below 30?"

print(f"Question: {question}")
print("─" * 55)

# With RAG
print("\nWITH RAG (grounded in knowledge base):")
rag_result = answer_question(question, use_rag=True)
print(f"\n{rag_result['answer']}")
print(f"\nSources used: {rag_result['sources']}")

time.sleep(5)

# Without RAG
print("\n" + "─" * 55)
print("WITHOUT RAG (LLM memory only):")
import time
no_rag_result = answer_question(question, use_rag=False)
print(f"\n{no_rag_result['answer']}")

print("\n" + "─" * 55)
print("KEY DIFFERENCE:")
print("RAG answer is grounded in InvestIQ's specific")
print("interpretation of RSI, consistent with the model.")
print("No-RAG answer comes from general LLM knowledge")
print("which may not match InvestIQ's specific logic.")

Question: What does it mean when RSI is below 30?
───────────────────────────────────────────────────────

WITH RAG (grounded in knowledge base):
Knowledge base loaded: 49 documents.
Knowledge base loaded: 49 documents.

When the Relative Strength Index (RSI) is below 30, it indicates that a stock may be "oversold." According to the InvestIQ knowledge base, this means the stock has fallen too far and too fast, suggesting it could potentially bounce back up. 

RSI is a measurement between 0 and 100 that tracks how quickly and by how much a stock's price has moved recently. While this level suggests a potential shift in momentum, please note that this information is for educational purposes to help you understand market concepts.

Sources used: ['What is RSI (Relative Strength Index)?']

───────────────────────────────────────────────────────
WITHOUT RAG (LLM memory only):

In technical analysis, the Relative Strength Index (RSI) is a tool used to measure the speed and change of price mo

Full Q&A demo

In [6]:
import time

questions = [
    "What is a Bollinger Band squeeze?",
    "Why does the 50-day moving average matter so much?",
    "What does InvestIQ's probability score actually mean?",
    "How does InvestIQ explain its predictions?",
    "What is ATR and why should I care about it?",
]

print("InvestIQ Q&A System Demo")
print("═" * 55)

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(5)   # rate limit protection

    result = answer_question(q, ticker="RELIANCE.NS")

    print(f"\nQ: {q}")
    print(f"{'─'*55}")
    print(f"A: {result['answer']}")
    print(f"\n   Sources: {result['sources']}")

InvestIQ Q&A System Demo
═══════════════════════════════════════════════════════
Knowledge base loaded: 49 documents.
Knowledge base loaded: 49 documents.

Q: What is a Bollinger Band squeeze?
───────────────────────────────────────────────────────
A: A Bollinger Band squeeze occurs when the three lines drawn around a stock's price chart—the middle 20-day average and the upper and lower bands—become very narrow. This indicates that the stock's volatility is unusually low. 

Historically, these squeezes are often followed by large price moves. However, the provided context does not specify which direction the price will move; the direction remains unknown until the move actually begins. InvestIQ monitors the band width specifically to detect when these squeeze conditions are occurring for stocks like RELIANCE.NS.

   Sources: ['What are Bollinger Bands?']
Knowledge base loaded: 49 documents.
Knowledge base loaded: 49 documents.

Q: Why does the 50-day moving average matter so much?
────

Inspect retrieved context

In [7]:
# Show exactly what the model is reading before answering
question = "Why is MACD important in InvestIQ?"

print(f"Question: {question}")
print("\nRetrieved context (what Gemini reads):")
print("─" * 55)

chunks = retrieve(question, n_results=3)
for i, chunk in enumerate(chunks):
    print(f"\n[Chunk {i+1}] Source: {chunk['title']}")
    print(f"Relevance distance: {chunk['distance']:.4f}")
    print(f"Content preview: {chunk['content'][:200]}...")

Question: Why is MACD important in InvestIQ?

Retrieved context (what Gemini reads):
───────────────────────────────────────────────────────
Knowledge base loaded: 49 documents.

[Chunk 1] Source: What is MACD?
Relevance distance: 0.3337
Content preview: The MACD histogram is the most important part for InvestIQ. It
shows whether momentum is accelerating or decelerating. A growing
histogram means momentum is building. A shrinking histogram is
an early...

[Chunk 2] Source: How does InvestIQ explain its predictions?
Relevance distance: 0.4472
Content preview: This transparency is what makes InvestIQ different from a black
box. You never just get a number — you always see the reasoning
behind it....

[Chunk 3] Source: What is MACD?
Relevance distance: 0.4608
Content preview: MACD stands for Moving Average Convergence Divergence. It sounds
complicated but the idea is simple: it measures whether a stock's
short-term momentum is stronger or weaker than its longer-term
moment...


Stock-specific Q&A

In [8]:
import time

# Demonstrate that questions can be answered in context of a ticker
stock_questions = [
    ("RELIANCE.NS", "Why is the model's confidence LOW for Reliance?"),
    ("TCS.NS",      "What does high RSI mean for TCS right now?"),
    ("HDFCBANK.NS", "What should I know about HDFC Bank's signals?"),
]

print("Stock-specific Q&A")
print("═" * 55)

for ticker, question in stock_questions:
    result = answer_question(question, ticker=ticker)
    print(f"\n[{ticker}] {question}")
    print(f"{'─'*50}")
    print(result["answer"])
    time.sleep(5)

Stock-specific Q&A
═══════════════════════════════════════════════════════
Knowledge base loaded: 49 documents.
Knowledge base loaded: 49 documents.

[RELIANCE.NS] Why is the model's confidence LOW for Reliance?
──────────────────────────────────────────────────
When the InvestIQ model shows a LOW confidence score for Reliance, it means the probability is close to 50%. This does not indicate that the model is broken; rather, it suggests that the current market signals are mixed or contradictory. 

Because no model can predict the market with certainty, a LOW confidence signal is a cue to wait for clearer information before making any decisions. Please remember that InvestIQ should be used as only one input in your research, not as the sole basis for your investment choices. The provided context does not specify the exact data points causing the uncertainty for Reliance, only the general meaning of the score.
Knowledge base loaded: 49 documents.
Knowledge base loaded: 49 documents.

[TC

Summary

In [9]:
print("═" * 55)
print("WEEK 9 COMPLETE — RAG Q&A SYSTEM")
print("═" * 55)
print(f"""
Knowledge base:
  Documents  : {len(DOCUMENTS)}
  Topics     : {list(TOPICS.keys())}
  Vector DB  : ChromaDB (local, persistent)
  Embeddings : all-MiniLM-L6-v2 (384-dim)

How it works:
  1. User question is embedded into a 384-dim vector
  2. ChromaDB finds most similar document chunks
  3. Retrieved chunks are injected into Gemini prompt
  4. Gemini answers ONLY from retrieved context
  5. Sources are cited alongside the answer

Why RAG over pure LLM:
  - Answers are grounded in InvestIQ's specific logic
  - Consistent with how the model actually works
  - Hallucination risk dramatically reduced
  - Every answer is citable and verifiable

In Week 10 (FastAPI), this becomes the /ask endpoint:
  GET /ask?q=what+is+RSI&ticker=RELIANCE.NS
  → returns answer + sources as JSON
""")

═══════════════════════════════════════════════════════
WEEK 9 COMPLETE — RAG Q&A SYSTEM
═══════════════════════════════════════════════════════

Knowledge base:
  Documents  : 10
  Topics     : ['RSI', 'MACD', 'Bollinger Bands', 'Moving Averages', 'ATR', 'Volume', 'Probability', 'SHAP', 'News Sentiment', 'NIFTY50']
  Vector DB  : ChromaDB (local, persistent)
  Embeddings : all-MiniLM-L6-v2 (384-dim)

How it works:
  1. User question is embedded into a 384-dim vector
  2. ChromaDB finds most similar document chunks
  3. Retrieved chunks are injected into Gemini prompt
  4. Gemini answers ONLY from retrieved context
  5. Sources are cited alongside the answer

Why RAG over pure LLM:
  - Answers are grounded in InvestIQ's specific logic
  - Consistent with how the model actually works
  - Hallucination risk dramatically reduced
  - Every answer is citable and verifiable

In Week 10 (FastAPI), this becomes the /ask endpoint:
  GET /ask?q=what+is+RSI&ticker=RELIANCE.NS
  → returns answer +